# SC-Flow vs ContextFlow — Side-by-Side Comparison

Compares 100 runs of **sc-flow-tools** (CTF-H coupling) against 100 runs of the
original **ContextFlow** implementation on the GSE232025 axolotl dataset.

**Conditions compared:**
- `extrap_100` — Extrapolation (trained on Stage44→Stage54→Stage57→Juvenile, predict Adult)
- `intrap_100` — Interpolation (trained on Stage44→Stage54→Stage57→Adult, predict Juvenile)

**Metrics:** Sliced Wasserstein (W2), MMD, Energy distance  
**Prediction modes:** Next-Step (NS) and IVP (full trajectory from t=0 to t_end)

Each panel shows: **mean ± std** across 100 seeds, per transition.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
SCFLOW_RESULTS  = "results"                  # relative to this notebook
CTF_BASE        = "/scratch/icbb/vishak/contextflow/contextflow_orig/ContextFlow/experiment_figures/use_all_data_False_GSE232025"

N_RUNS = 100

TRANSITIONS = [
    "Stage44 -> Stage54",
    "Stage54 -> Stage57",
    "Stage57 -> Juvenile",
    "Juvenile -> Adult",
]
N_TRANS = len(TRANSITIONS)

METRICS   = ["W2", "MMD", "Energy"]
COLORS    = {"W2": "#2196F3", "MMD": "#FF5722", "Energy": "#4CAF50"}
MARKERS   = {"W2": "o",       "MMD": "s",       "Energy": "^"}  

## Data Loaders

In [ ]:
def load_scflow_runs(condition: str, csv_name: str) -> dict[str, np.ndarray]:
    """
    Load sc-flow-tools CSV results across N_RUNS runs.

    Returns dict: metric -> array of shape (N_TRANS, n_valid_runs)
    """
    data = {m: [] for m in METRICS}
    base = os.path.join(SCFLOW_RESULTS, condition)
    for run_id in range(N_RUNS):
        path = os.path.join(base, f"run_{run_id}", csv_name)
        if not os.path.exists(path):
            continue
        df = pd.read_csv(path)
        # Sort by TRANSITIONS order
        df = df.set_index("transition").reindex(TRANSITIONS)
        if df.isnull().all().any():
            continue
        data["W2"].append(df["W2 (model)"].values)
        data["MMD"].append(df["MMD (model)"].values)
        data["Energy"].append(df["Energy (model)"].values)
    return {m: np.array(v).T for m, v in data.items()}  # shape (N_TRANS, n_runs)


def load_ctf_runs(prefix: str, json_name: str, ivp: bool = False) -> dict[str, np.ndarray]:
    """
    Load ContextFlow JSON results from folders named f'{prefix}_{i}' for i in 1..100.

    For IVP, the JSON has 5 values (index 0 = trivial Stage44 start), so we take [1:5].
    For NS,  the JSON has 4 values corresponding directly to TRANSITIONS.

    Returns dict: metric -> array of shape (N_TRANS, n_valid_runs)
    """
    CTF_METRIC_MAP = {"W2": "wasserstein", "MMD": "mmd", "Energy": "energy"}
    data = {m: [] for m in METRICS}
    for i in range(1, N_RUNS + 1):
        path = os.path.join(CTF_BASE, f"{prefix}_{i}", "Default", json_name)
        if not os.path.exists(path):
            continue
        with open(path) as f:
            j = json.load(f)
        try:
            for m in METRICS:
                key = CTF_METRIC_MAP[m]
                vals = j[key]
                if ivp:
                    vals = vals[1:5]   # skip trivial t=0 entry
                if len(vals) != N_TRANS:
                    continue
                data[m].append(vals)
        except (KeyError, IndexError):
            continue
    return {m: np.array(v).T for m, v in data.items()}  # shape (N_TRANS, n_runs)

In [ ]:
# ── Load everything ──────────────────────────────────────────────────────────
print("Loading sc-flow-tools results...")
sf_extrap_ns  = load_scflow_runs("extrapolation",  "metrics_ns.csv")
sf_extrap_ivp = load_scflow_runs("extrapolation",  "metrics_ivp.csv")
sf_interp_ns  = load_scflow_runs("interpolation",  "metrics_ns.csv")
sf_interp_ivp = load_scflow_runs("interpolation",  "metrics_ivp.csv")

print("Loading ContextFlow results...")
ctf_extrap_ns  = load_ctf_runs("extrap_100", "next_step_error.json", ivp=False)
ctf_extrap_ivp = load_ctf_runs("extrap_100", "IVP_error.json",       ivp=True)
ctf_interp_ns  = load_ctf_runs("interp_100", "next_step_error.json", ivp=False)
ctf_interp_ivp = load_ctf_runs("interp_100", "IVP_error.json",       ivp=True)

# Print run counts
for label, d in [
    ("SF extrap NS",  sf_extrap_ns),
    ("SF extrap IVP", sf_extrap_ivp),
    ("SF interp NS",  sf_interp_ns),
    ("SF interp IVP", sf_interp_ivp),
    ("CTF extrap NS",  ctf_extrap_ns),
    ("CTF extrap IVP", ctf_extrap_ivp),
    ("CTF interp NS",  ctf_interp_ns),
    ("CTF interp IVP", ctf_interp_ivp),
]:
    n = d["W2"].shape[1] if d["W2"].size else 0
    print(f"  {label}: {n} runs")

## Plotting Functions

In [ ]:
def plot_comparison(ax, sf_data: dict, ctf_data: dict, metric: str, title: str):
    """
    Plot mean ± std for one metric comparing sc-flow (solid) vs contextflow (dashed)
    on a single axes object.
    """
    x = np.arange(N_TRANS)
    xlabels = [t.replace(" -> ", "\n→ ") for t in TRANSITIONS]
    color = COLORS[metric]
    marker = MARKERS[metric]

    # sc-flow-tools
    sf_vals = sf_data[metric]          # (N_TRANS, n_runs)
    sf_mean = sf_vals.mean(axis=1)
    sf_std  = sf_vals.std(axis=1)
    ax.plot(x, sf_mean, color=color, lw=2, marker=marker, ms=6,
            linestyle="-", label="sc-flow-tools")
    ax.fill_between(x, sf_mean - sf_std, sf_mean + sf_std,
                    color=color, alpha=0.15)

    # contextflow
    ctf_vals = ctf_data[metric]
    ctf_mean = ctf_vals.mean(axis=1)
    ctf_std  = ctf_vals.std(axis=1)
    ax.plot(x, ctf_mean, color=color, lw=2, marker=marker, ms=6,
            linestyle="--", label="ContextFlow")
    ax.fill_between(x, ctf_mean - ctf_std, ctf_mean + ctf_std,
                    color=color, alpha=0.08)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel(metric)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    ax.grid(axis="y", which="minor", linestyle=":", alpha=0.2)


def plot_all_metrics_row(axes_row, sf_data, ctf_data, row_title: str):
    """Fill one row of axes (3 panels: W2, MMD, Energy)."""
    for ax, metric in zip(axes_row, METRICS):
        plot_comparison(ax, sf_data, ctf_data, metric,
                        title=f"{row_title} — {metric}")


def add_legend(fig):
    handles = [
        Line2D([0], [0], color="grey", lw=2, linestyle="-",  label="sc-flow-tools"),
        Line2D([0], [0], color="grey", lw=2, linestyle="--", label="ContextFlow"),
    ]
    fig.legend(handles=handles, loc="lower center", ncol=2,
               fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))

## Extrapolation — NS vs IVP (all 3 metrics)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey="row")

plot_all_metrics_row(axes[0], sf_extrap_ns,  ctf_extrap_ns,  "Extrapolation — Next-Step")
plot_all_metrics_row(axes[1], sf_extrap_ivp, ctf_extrap_ivp, "Extrapolation — IVP")

fig.suptitle(
    "Extrapolation — sc-flow-tools vs ContextFlow (100 runs, mean ± std)",
    fontsize=13, fontweight="bold", y=1.01
)
add_legend(fig)
fig.tight_layout()
plt.savefig(os.path.join(SCFLOW_RESULTS, "comparison_extrapolation.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/comparison_extrapolation.png")

## Interpolation — NS vs IVP (all 3 metrics)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey="row")

plot_all_metrics_row(axes[0], sf_interp_ns,  ctf_interp_ns,  "Interpolation — Next-Step")
plot_all_metrics_row(axes[1], sf_interp_ivp, ctf_interp_ivp, "Interpolation — IVP")

fig.suptitle(
    "Interpolation — sc-flow-tools vs ContextFlow (100 runs, mean ± std)",
    fontsize=13, fontweight="bold", y=1.01
)
add_legend(fig)
fig.tight_layout()
plt.savefig(os.path.join(SCFLOW_RESULTS, "comparison_interpolation.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/comparison_interpolation.png")

## Combined 4-panel Overview

One figure: rows = (Extrap, Interp) × cols = (NS, IVP), single metric W2 for quick glance.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=False)

combos = [
    (axes[0, 0], sf_extrap_ns,  ctf_extrap_ns,  "Extrapolation — Next-Step"),
    (axes[0, 1], sf_extrap_ivp, ctf_extrap_ivp, "Extrapolation — IVP"),
    (axes[1, 0], sf_interp_ns,  ctf_interp_ns,  "Interpolation — Next-Step"),
    (axes[1, 1], sf_interp_ivp, ctf_interp_ivp, "Interpolation — IVP"),
]

for ax, sf, ctf, title in combos:
    x = np.arange(N_TRANS)
    xlabels = [t.replace(" -> ", "\n→ ") for t in TRANSITIONS]
    for metric in METRICS:
        color  = COLORS[metric]
        marker = MARKERS[metric]

        sf_mean = sf[metric].mean(axis=1)
        sf_std  = sf[metric].std(axis=1)
        ax.plot(x, sf_mean, color=color, lw=2, marker=marker, ms=5,
                linestyle="-", label=f"{metric} sf")
        ax.fill_between(x, sf_mean - sf_std, sf_mean + sf_std,
                        color=color, alpha=0.12)

        ctf_mean = ctf[metric].mean(axis=1)
        ctf_std  = ctf[metric].std(axis=1)
        ax.plot(x, ctf_mean, color=color, lw=2, marker=marker, ms=5,
                linestyle="--", label=f"{metric} ctf")
        ax.fill_between(x, ctf_mean - ctf_std, ctf_mean + ctf_std,
                        color=color, alpha=0.06)

    ax.set_xticks(x)
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_ylabel("Metric value")
    ax.grid(axis="y", linestyle="--", alpha=0.35)

# Shared legend
metric_handles = [Line2D([0], [0], color=COLORS[m], lw=2, marker=MARKERS[m], ms=5, label=m)
                  for m in METRICS]
style_handles  = [
    Line2D([0], [0], color="grey", lw=2, linestyle="-",  label="sc-flow-tools"),
    Line2D([0], [0], color="grey", lw=2, linestyle="--", label="ContextFlow"),
]
fig.legend(handles=metric_handles + style_handles, loc="lower center",
           ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    "SC-Flow vs ContextFlow — 100 runs, mean ± std",
    fontsize=13, fontweight="bold", y=1.01
)
fig.tight_layout()
plt.savefig(os.path.join(SCFLOW_RESULTS, "comparison_overview.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/comparison_overview.png")

## Summary Table

In [ ]:
def summary_table(sf_data, ctf_data, label: str):
    rows = []
    for i, trans in enumerate(TRANSITIONS):
        for metric in METRICS:
            sf_vals  = sf_data[metric][i]
            ctf_vals = ctf_data[metric][i]
            rows.append({
                "Condition": label,
                "Transition": trans,
                "Metric": metric,
                "SC-Flow mean": np.mean(sf_vals).round(4),
                "SC-Flow std":  np.std(sf_vals).round(4),
                "CTF mean":     np.mean(ctf_vals).round(4),
                "CTF std":      np.std(ctf_vals).round(4),
                "Δ (SF - CTF)": (np.mean(sf_vals) - np.mean(ctf_vals)).round(4),
            })
    return pd.DataFrame(rows)


tables = pd.concat([
    summary_table(sf_extrap_ns,  ctf_extrap_ns,  "Extrap NS"),
    summary_table(sf_extrap_ivp, ctf_extrap_ivp, "Extrap IVP"),
    summary_table(sf_interp_ns,  ctf_interp_ns,  "Interp NS"),
    summary_table(sf_interp_ivp, ctf_interp_ivp, "Interp IVP"),
], ignore_index=True)

with pd.option_context("display.max_rows", 80, "display.max_columns", 10,
                       "display.width", 120):
    display(tables)

tables.to_csv(os.path.join(SCFLOW_RESULTS, "comparison_summary.csv"), index=False)
print("Saved: results/comparison_summary.csv")